In [3]:
!pip install grandalf langchain langgraph langchain-community -q

In [6]:
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. 상태 정의
class AgentState(TypedDict):
    query: str               # 사용자 질의
    symptoms: str            # 추출된 신체 증상/상태
    exercise_candidates: str # 운동 후보 리스트
    result: str              # 최종 답변

# 2. LLM 초기화
llm = Ollama(model="exaone3.5:2.4b")

# 3. 에이전트 정의

# -- 추출 에이전트: 질문에서 신체 상태/증상 키워드 추출
extractor_prompt = PromptTemplate.from_template("""
사용자의 질문에서 신체 상태나 건강 증상에 해당하는 단어 또는 구를 추출하세요.
결과는 쉼표로 구분된 문자열로만 출력하세요. 설명은 하지 마세요.
질문: {query}
""")

def extractor_agent(state: AgentState):
    chain = extractor_prompt | llm
    symptoms = chain.invoke({"query": state["query"]})
    return {**state, "symptoms": symptoms.strip()}

# -- 후보 에이전트: 증상을 해결할 수 있는 운동 리스트 추출
matcher_prompt = PromptTemplate.from_template("""
다음 신체 상태를 개선하는 데 효과적인 운동 5가지를 추천하세요.
결과는 운동 이름만 쉼표로 구분하여 출력하세요. 설명은 하지 마세요.
신체 상태: {symptoms}
""")

def matcher_agent(state: AgentState):
    chain = matcher_prompt | llm
    candidates = chain.invoke({"symptoms": state["symptoms"]})
    return {**state, "exercise_candidates": candidates.strip()}

# -- 답변 에이전트: 증상과 운동 리스트를 개조식으로 출력 (AI 냄새 없이)
answer_prompt = PromptTemplate.from_template("""
아래 정보를 바탕으로 운동 추천 답변을 작성하세요.

신체 상태: {symptoms}
추천 운동 목록: {exercise_candidates}

작성 규칙:
- 마크다운 기호(**, ##, --- 등) 사용 금지
- 이모지 사용 금지
- 딱딱한 AI 말투 금지 ("~드립니다", "~해드릴게요" 금지)
- 자연스러운 구어체로 작성
- 개조식(짧은 항목 나열)으로 작성
- 각 운동마다 이름, 방법 한 줄, 효과 한 줄로 구성
- 전체 분량은 10줄 내외로 간결하게
""")

def answer_agent(state: AgentState):
    chain = answer_prompt | llm
    answer = chain.invoke({
        "symptoms": state["symptoms"],
        "exercise_candidates": state["exercise_candidates"]
    })
    return {**state, "result": answer.strip()}

# 4. LangGraph 정의
graph = StateGraph(AgentState)
graph.add_node("extractor", extractor_agent)
graph.add_node("matcher",   matcher_agent)
graph.add_node("answer",    answer_agent)

graph.set_entry_point("extractor")
graph.add_edge("extractor", "matcher")
graph.add_edge("matcher",   "answer")
graph.add_edge("answer",    END)

app = graph.compile()

print("============================== LangGraph 구조:")
app.get_graph().print_ascii()

# 5. 실행
query = "체력이 안 좋고 살이 계속 찌는데 어떤 운동을 할까?"
result = app.invoke({"query": query})

print("============================== 추출된 증상:")
print(result["symptoms"])
print("============================== 추천 운동 목록:")
print(result["exercise_candidates"])
print("============================== 최종 답변:")
print(result["result"])

============================== LangGraph 구조:
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| extractor |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | matcher |   
 +---------+   
      *        
      *        
      *        
  +--------+   
  | answer |   
  +--------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   
============================== 추출된 증상:
체력, 살 찌는
============================== 추천 운동 목록:
심폐지구력 향상 운동, 저항성 훈련, 유산소 운동, 균형 잡힌 식단 조절, 고강도 인터벌 트레이닝
============================== 최종 답변:
## 운동 추천 리스트

### 심폐지구력 향상 운동
**버피 테스트**  
*유산소 능력 향상 및 심폐 기능 강화*

### 저항성 훈련
**덤벨 스니칭**  
*주요 근육 강화로 탄탄한 몸매 구축*

### 유산소 운동
**달리기**  
*유산소 활동으로 지방 소모 증가 및 체력 향상*

### 균형 잡힌 식단 조절
**단백질 위주 식단**  
*근육 회복 촉진 및 체중 관리 용이*

### 고강도 인터벌 트레이닝 (HIIT)
**격렬 달리기 + 점프 스쿼트**  
*짧은 시간 내 큰 칼로리 소모 및 효율적인 칼로리 감소*

이러한 운동들을 꾸준히 실천하면 체력 향상과 체중 관리에 큰 도움이 될 거예